In [1]:
# ============================================================
# 🏛️ Supreme Court Judgments Downloader (2020–2025)
# Downloads .txt + .json files from AWS Open Data (India)
# Saves year-wise into: D:\LPA_MTech_Project\My_Datasets\SC_2020-2025
# ============================================================

In [2]:
import os
import boto3
import botocore
import json
import pandas as pd
from tqdm import tqdm


In [3]:
# ---------- CONFIG ----------
LOCAL_SAVE_DIR = r"D:\LPA_MTech_Project\My_Datasets\SC_2010-2019"
START_YEAR = 2010
END_YEAR = 2019
BUCKET_NAME = "indian-supreme-court-judgments"
# ----------------------------

In [4]:
os.makedirs(LOCAL_SAVE_DIR, exist_ok=True)

In [5]:
# Anonymous S3 client (public dataset)
# s3 = boto3.client("s3", config=boto3.session.Config(signature_version="unsigned"))

s3 = boto3.client(
    "s3",
    config=botocore.client.Config(signature_version=botocore.UNSIGNED)
)


In [6]:
def list_zip_files_in_year(year):
    """List zip files (English + metadata) for each year."""
    prefixes = [
        f"data/zip/year={year}/",
        f"metadata/zip/year={year}/"
    ]
    for prefix in prefixes:
        paginator = s3.get_paginator("list_objects_v2")
        for page in paginator.paginate(Bucket=BUCKET_NAME, Prefix=prefix):
            for obj in page.get("Contents", []):
                key = obj["Key"]
                if key.endswith(".zip"):
                    yield key

In [7]:

def download_zip_file(key, local_path):
    """Download one zip file if not already downloaded."""
    if os.path.exists(local_path):
        return
    os.makedirs(os.path.dirname(local_path), exist_ok=True)
    try:
        s3.download_file(BUCKET_NAME, key, local_path)
    except Exception as e:
        print(f"❌ Error downloading {key}: {e}")

In [8]:
# ---------- DOWNLOAD SECTION ----------
for year in range(START_YEAR, END_YEAR + 1):
    print(f"\n📦 Downloading ZIP files for {year} ...")
    year_dir = os.path.join(LOCAL_SAVE_DIR, str(year))
    os.makedirs(year_dir, exist_ok=True)

    keys = list(list_zip_files_in_year(year))
    print(f"Found {len(keys)} zip files for {year}")

    for key in tqdm(keys, desc=f"Year {year}", unit="zip"):
        filename = key.split("/")[-1]
        local_path = os.path.join(year_dir, filename)
        download_zip_file(key, local_path)

print("\n✅ ZIP downloads complete for all years.")


📦 Downloading ZIP files for 2010 ...
Found 3 zip files for 2010


Year 2010: 100%|██████████| 3/3 [01:43<00:00, 34.66s/zip]



📦 Downloading ZIP files for 2011 ...
Found 3 zip files for 2011


Year 2011: 100%|██████████| 3/3 [01:34<00:00, 31.42s/zip]



📦 Downloading ZIP files for 2012 ...
Found 3 zip files for 2012


...


📦 Downloading ZIP files for 2018 ...
Found 3 zip files for 2018


Year 2018: 100%|██████████| 3/3 [01:29<00:00, 29.87s/zip]



📦 Downloading ZIP files for 2019 ...
Found 3 zip files for 2019


Year 2019: 100%|██████████| 3/3 [01:39<00:00, 33.24s/zip]


✅ ZIP downloads complete for all years.


In [9]:
# ---------- OPTIONAL: Extract ZIPs ----------
import zipfile

for year in range(START_YEAR, END_YEAR + 1):
    year_dir = os.path.join(LOCAL_SAVE_DIR, str(year))
    print(f"\n📂 Extracting ZIPs in {year_dir}")
    for file in os.listdir(year_dir):
        if file.endswith(".zip"):
            zip_path = os.path.join(year_dir, file)
            extract_dir = os.path.join(year_dir, file.replace(".zip", ""))
            if not os.path.exists(extract_dir):
                os.makedirs(extract_dir, exist_ok=True)
                with zipfile.ZipFile(zip_path, 'r') as zf:
                    zf.extractall(extract_dir)
print("\n✅ All ZIPs extracted successfully!")


📂 Extracting ZIPs in D:\LPA_MTech_Project\My_Datasets\SC_2010-2019\2010

📂 Extracting ZIPs in D:\LPA_MTech_Project\My_Datasets\SC_2010-2019\2011

📂 Extracting ZIPs in D:\LPA_MTech_Project\My_Datasets\SC_2010-2019\2012

📂 Extracting ZIPs in D:\LPA_MTech_Project\My_Datasets\SC_2010-2019\2013

📂 Extracting ZIPs in D:\LPA_MTech_Project\My_Datasets\SC_2010-2019\2014

📂 Extracting ZIPs in D:\LPA_MTech_Project\My_Datasets\SC_2010-2019\2015

📂 Extracting ZIPs in D:\LPA_MTech_Project\My_Datasets\SC_2010-2019\2016

📂 Extracting ZIPs in D:\LPA_MTech_Project\My_Datasets\SC_2010-2019\2017

📂 Extracting ZIPs in D:\LPA_MTech_Project\My_Datasets\SC_2010-2019\2018

📂 Extracting ZIPs in D:\LPA_MTech_Project\My_Datasets\SC_2010-2019\2019

✅ All ZIPs extracted successfully!


In [10]:
# ============================================================
# 📊 METADATA CSV BUILDER
# ============================================================


In [11]:
metadata_records = []

In [12]:
for year in range(START_YEAR, END_YEAR + 1):
    year_dir = os.path.join(LOCAL_SAVE_DIR, str(year))
    print(f"\n🧩 Extracting metadata from {year_dir}")
    
    # Look inside each extracted metadata folder
    for root, _, files in os.walk(year_dir):
        for file in files:
            if file.endswith(".json"):
                file_path = os.path.join(root, file)
                try:
                    with open(file_path, "r", encoding="utf-8") as f:
                        data = json.load(f)

                    record = {
                        "year": year,
                        "file_name": file,
                        "case_name": data.get("case_name") or data.get("title") or "",
                        "date": data.get("date") or data.get("judgment_date") or "",
                        "bench": data.get("bench") or "",
                        "judges": ", ".join(data.get("judges", [])) if isinstance(data.get("judges"), list) else data.get("judges", ""),
                        "citations": ", ".join(data.get("citations", [])) if isinstance(data.get("citations"), list) else data.get("citations", ""),
                        "verdict_summary": data.get("verdict_summary", ""),
                        "url": data.get("url") or data.get("link") or "",
                    }
                    metadata_records.append(record)
                except Exception as e:
                    print(f"⚠️ Error reading {file}: {e}")


🧩 Extracting metadata from D:\LPA_MTech_Project\My_Datasets\SC_2010-2019\2010

🧩 Extracting metadata from D:\LPA_MTech_Project\My_Datasets\SC_2010-2019\2011

🧩 Extracting metadata from D:\LPA_MTech_Project\My_Datasets\SC_2010-2019\2012

🧩 Extracting metadata from D:\LPA_MTech_Project\My_Datasets\SC_2010-2019\2013

🧩 Extracting metadata from D:\LPA_MTech_Project\My_Datasets\SC_2010-2019\2014

🧩 Extracting metadata from D:\LPA_MTech_Project\My_Datasets\SC_2010-2019\2015

🧩 Extracting metadata from D:\LPA_MTech_Project\My_Datasets\SC_2010-2019\2016

🧩 Extracting metadata from D:\LPA_MTech_Project\My_Datasets\SC_2010-2019\2017

🧩 Extracting metadata from D:\LPA_MTech_Project\My_Datasets\SC_2010-2019\2018

🧩 Extracting metadata from D:\LPA_MTech_Project\My_Datasets\SC_2010-2019\2019


In [13]:
# Create DataFrame
metadata_df = pd.DataFrame(metadata_records)
csv_path = os.path.join(LOCAL_SAVE_DIR, "cases_metadata_2010_2019.csv")
metadata_df.to_csv(csv_path, index=False, encoding="utf-8-sig")

In [14]:
print(f"\n✅ Metadata CSV created successfully: {csv_path}")
print(f"Total cases processed: {len(metadata_df)}")
metadata_df.head()


✅ Metadata CSV created successfully: D:\LPA_MTech_Project\My_Datasets\SC_2010-2019\cases_metadata_2010_2019.csv
Total cases processed: 7952


,year,file_name,case_name,date,bench,judges,citations,verdict_summary,url
0,2010,2010_10_1002_1008.json,,,,,,,
1,2010,2010_10_1009_1021.json,,,,,,,
2,2010,2010_10_1022_1069.json,,,,,,,
3,2010,2010_10_1070_1094.json,,,,,,,
4,2010,2010_10_108_123.json,,,,,,,
